# LV Importance - GTEx True Tissue Labels (ARCHS4 projection)

Train one binary Random Forest per GTEx tissue using:
- **Features**: LV scores from the ARCHS4→GTEx projection (B matrix, samples × LVs)
- **Labels**: true GTEx tissue labels (`SMTS`) from the GTEx metadata

## Imports & config

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from sklearn.metrics import balanced_accuracy_score, precision_score

import shap

from pyprojroot.here import here
import re
import textwrap
from matplotlib.colors import Normalize, LinearSegmentedColormap
from matplotlib.lines import Line2D

import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
from rpy2.robjects.conversion import localconverter

readRDS = ro.r["readRDS"]


/home/msubirana/miniconda3/envs/clamp-analyses/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Thresholds
MIN_BALANCED_ACC = 0.90   # balanced accuracy on held-out test fold
MIN_PRECISION    = 0.90   # precision for the positive (tissue) class
MIN_SAMPLES      = 50     # skip tissue if fewer samples
CUMULATIVE_PCT   = 25     # top LVs that cover this % of positive SHAP
GLOBAL_SEED      = 42

PARAM_GRID = {
    "n_estimators":      [100, 300, 500, 1000],
    "max_depth":         [None, 10, 20, 30],
    "min_samples_split": [2, 5, 10],
}

# Paths
PROJ_CSV = here() / "output" / "03_model_biology" / "00_archs4" / "03_tissue_projections" / "00_gtex_archs4_projection" / "gtex_archs4_projection.csv"
META_TXT = here() / "data" / "gtex" / "GTEx_Analysis_v8_Annotations_SampleAttributesDS.txt"
OUT_DIR  = here() / "output" / "03_model_biology" / "00_archs4" / "03_tissue_projections" / "01_gtex_archs4_projection_LV_importance"
OUT_DIR.mkdir(parents=True, exist_ok=True)
(OUT_DIR / "per_tissue").mkdir(exist_ok=True)
print("Output dir:", OUT_DIR)

ACC_PATH = OUT_DIR / "accuracy_summary.tsv"
TOP_LV_PATH = OUT_DIR / "top_lvs_cumulative.tsv"
BIOLOGY_PATH = OUT_DIR / "biology_top_lvs.tsv"
UNIQUE_TOP_LV_PATH = OUT_DIR / "top_lvs_cumulative_unique.tsv"
TRAITS_PATH = here() / "data" / "archs4" / "traits" / "hall_coverage_rs100_seed_1_CLAMPfull_hall" / "gls-summary-phenomexcan.tsv.gz"
PHENO_INFO_PATH = here() / "data" / "gtex" / "phenomexcan_simplified_phenotypes_info.tsv.gz"
DOTPLOT_DIR = OUT_DIR / "dotplots"
DOTPLOT_DIR.mkdir(exist_ok=True)

MODEL_RDS = here() / "output" / "01_model_building" / "04_archs4" / "06_bp_coverage_rshall" / "06_bp_coverage_hall_rs_100" / "hall_coverage_rs100_seed_1" / "CLAMPfull_hall.rds"


Output dir: /home/msubirana/Documents/pivlab/clamp-analyses/output/03_model_biology/00_archs4/03_tissue_projections/01_gtex_archs4_projection_LV_importance/archs4_feature_importance_gtex_true_labels


## Load projection (B matrix) and GTEx metadata

In [3]:
# Projection saved as LVs x samples  ->  transpose to samples x LVs
proj_raw = pd.read_csv(PROJ_CSV, index_col=0)
print("Raw projection shape (LVs x samples):", proj_raw.shape)

lv_matrix = proj_raw.T.copy()
lv_matrix.index.name = "SAMPID"
print("LV matrix (samples x LVs):", lv_matrix.shape)

Raw projection shape (LVs x samples): (363, 17382)
LV matrix (samples x LVs): (17382, 363)


In [4]:
# True tissue labels from GTEx metadata
meta = pd.read_csv(
    META_TXT,
    sep="\t",
    usecols=["SAMPID", "SMTS"],
    dtype=str,
).dropna(subset=["SMTS"]).set_index("SAMPID")

print("Metadata samples:", len(meta))
print("Unique SMTS (broad tissue):", meta["SMTS"].nunique())

Metadata samples: 22951
Unique SMTS (broad tissue): 31


In [5]:
# Align samples present in both the projection and the metadata
common_samples = lv_matrix.index.intersection(meta.index)
print(f"Aligned samples: {len(common_samples)} / {len(lv_matrix)} in projection")

lv_matrix     = lv_matrix.loc[common_samples]
tissue_labels = meta.loc[common_samples, "SMTS"]

# Keep only samples from tissues with >= MIN_SAMPLES (same filter as reference notebooks)
tissue_counts  = tissue_labels.value_counts()
valid_tissues  = tissue_counts[tissue_counts >= MIN_SAMPLES].index
mask           = tissue_labels.isin(valid_tissues)
lv_matrix      = lv_matrix.loc[mask]
tissue_labels  = tissue_labels.loc[mask]

print(f"After MIN_SAMPLES={MIN_SAMPLES} filter:")
print(f"  Samples : {len(lv_matrix)}")
print(f"  Tissues : {tissue_labels.nunique()}")
print(tissue_labels.value_counts().to_string())

Aligned samples: 17382 / 17382 in projection


After MIN_SAMPLES=50 filter:
  Samples : 17333
  Tissues : 27
SMTS
Brain              2642
Skin               1809
Esophagus          1445
Blood Vessel       1335
Adipose Tissue     1204
Blood               929
Heart               861
Muscle              803
Colon               779
Thyroid             653
Nerve               619
Lung                578
Breast              459
Testis              361
Stomach             359
Pancreas            328
Pituitary           283
Adrenal Gland       258
Prostate            245
Spleen              241
Liver               226
Small Intestine     187
Ovary               180
Salivary Gland      162
Vagina              156
Uterus              142
Kidney               89


In [6]:
model   = readRDS(str(MODEL_RDS))

with localconverter(ro.default_converter + pandas2ri.converter):
    lv_ann = ro.conversion.rpy2py(model.rx2("summary"))

lv_ann = lv_ann.reset_index(drop=True)
print("LV annotations from model summary:", lv_ann.shape)
lv_ann.head(3)

LV annotations from model summary: (10267, 7)


,pathway,LV,AUC,p_value,FDR,npos,nneg
0,REACTOME_REACTOME_ALPHA_LINOLENIC_OMEGA3_AND_L...,LV1,0.574943,0.356096,0.613156,2,18274
1,REACTOME_REACTOME_BIOTIN_TRANSPORT_AND_METABOLISM,LV1,0.598692,0.313539,0.578455,2,18274
2,REACTOME_REACTOME_MITOCHONDRIAL_TRNA_AMINOACYL...,LV1,0.771957,0.029149,0.116311,4,18274


## Binary Random Forest per tissue (true labels)

In [7]:
tissues_to_test = sorted(tissue_labels.unique().tolist())
print(f"Tissues to train RF on: {len(tissues_to_test)}")
print(tissues_to_test)

Tissues to train RF on: 27
['Adipose Tissue', 'Adrenal Gland', 'Blood', 'Blood Vessel', 'Brain', 'Breast', 'Colon', 'Esophagus', 'Heart', 'Kidney', 'Liver', 'Lung', 'Muscle', 'Nerve', 'Ovary', 'Pancreas', 'Pituitary', 'Prostate', 'Salivary Gland', 'Skin', 'Small Intestine', 'Spleen', 'Stomach', 'Testis', 'Thyroid', 'Uterus', 'Vagina']


In [ ]:
def tissue_to_shap_path(tissue):
    return OUT_DIR / "per_tissue" / f"shap_{tissue.replace(' ', '_')}.tsv"


def load_cached_rf_results():
    if not ACC_PATH.exists():
        return None, None

    df_cached_acc = pd.read_csv(ACC_PATH, sep="\t")
    if df_cached_acc["passes"].dtype != bool:
        df_cached_acc["passes"] = df_cached_acc["passes"].map({
            True: True,
            False: False,
            "True": True,
            "False": False,
        })
        if df_cached_acc["passes"].isna().any():
            raise ValueError("Could not parse cached passes column as booleans.")

    cached_passing = df_cached_acc.loc[df_cached_acc["passes"], "tissue"].tolist()
    missing_shap = [tissue_to_shap_path(tissue) for tissue in cached_passing
                    if not tissue_to_shap_path(tissue).exists()]
    if missing_shap:
        print("Cached accuracy summary exists, but SHAP files are missing:")
        for path in missing_shap:
            print("  ", path)
        return None, None

    cached_shap_store = {
        tissue: pd.read_csv(tissue_to_shap_path(tissue), sep="\t")
        for tissue in cached_passing
    }
    return df_cached_acc.sort_values("balanced_accuracy", ascending=False), cached_shap_store


df_acc, shap_store = load_cached_rf_results()

if df_acc is not None:
    print(f"Loaded cached RF/SHAP results from {OUT_DIR}")
else:
    print("Cached RF/SHAP results not found or incomplete; training random forests.")
    accuracy_records = []
    shap_store       = {}   # tissue -> df of positive SHAP values

    for tissue in tissues_to_test:
        y = (tissue_labels == tissue).astype(int)

        outer_cv       = StratifiedKFold(n_splits=5, shuffle=True, random_state=GLOBAL_SEED)
        fold_acc, fold_prec = [], []

        for fold, (dev_idx, test_idx) in enumerate(outer_cv.split(lv_matrix, y)):
            X_dev, X_test = lv_matrix.iloc[dev_idx], lv_matrix.iloc[test_idx]
            y_dev, y_test = y.iloc[dev_idx],         y.iloc[test_idx]

            inner_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=GLOBAL_SEED + fold)
            search   = RandomizedSearchCV(
                estimator          = RandomForestClassifier(random_state=GLOBAL_SEED, class_weight="balanced"),
                param_distributions= PARAM_GRID,
                n_iter             = 15,
                cv                 = inner_cv,
                scoring            = "balanced_accuracy",
                n_jobs             = -1,
                random_state       = GLOBAL_SEED + fold,
            )
            search.fit(X_dev, y_dev)

            y_pred = search.best_estimator_.predict(X_test)
            fold_acc.append(balanced_accuracy_score(y_test, y_pred))
            fold_prec.append(precision_score(y_test, y_pred, pos_label=1, zero_division=0))

        mean_acc  = float(np.mean(fold_acc))
        mean_prec = float(np.mean(fold_prec))
        passes    = (mean_acc >= MIN_BALANCED_ACC) and (mean_prec >= MIN_PRECISION)

        accuracy_records.append({
            "tissue":            tissue,
            "n_positive":        int(y.sum()),
            "n_negative":        int((y == 0).sum()),
            "balanced_accuracy": round(mean_acc,  4),
            "precision":         round(mean_prec, 4),
            "passes":            passes,
        })
        status = "PASS" if passes else "skip"
        print(f"[{status}]  {tissue:<35s}  acc={mean_acc:.3f}  prec={mean_prec:.3f}")

        if not passes:
            continue

        # Retrain on all samples with best hyperparams from last fold
        final_rf = RandomForestClassifier(
            random_state  = GLOBAL_SEED,
            class_weight  = "balanced",
            **search.best_params_,
        )
        final_rf.fit(lv_matrix, y)

        # SHAP values for the positive (tissue) class
        explainer   = shap.TreeExplainer(final_rf, data=lv_matrix, feature_perturbation="interventional")
        shap_values = explainer(lv_matrix).values[..., 1]          # class-1 SHAP

        tissue_mask = (y == 1).values
        mean_shap   = np.mean(shap_values[tissue_mask], axis=0)    # average over tissue samples

        df_shap = (
            pd.DataFrame({"LV": lv_matrix.columns, "mean_shap": mean_shap, "tissue": tissue})
            .query("mean_shap > 0")
            .sort_values("mean_shap", ascending=False)
            .reset_index(drop=True)
        )
        total = df_shap["mean_shap"].sum()
        df_shap["cumulative_pct"] = (df_shap["mean_shap"].cumsum() / total * 100)

        shap_store[tissue] = df_shap
        df_shap.to_csv(tissue_to_shap_path(tissue), sep="\t", index=False)

    df_acc = pd.DataFrame(accuracy_records).sort_values("balanced_accuracy", ascending=False)
    df_acc.to_csv(ACC_PATH, sep="\t", index=False)
    print("\nDone.")


In [ ]:
df_acc

## Top LVs per tissue (SHAP cumulative importance)

In [ ]:
passing_tissues = df_acc.loc[df_acc["passes"], "tissue"].tolist()
print(f"Tissues that passed filters: {len(passing_tissues)}")

if not passing_tissues:
    print("No tissue passed the thresholds.")
else:
    if TOP_LV_PATH.exists():
        df_top = pd.read_csv(TOP_LV_PATH, sep="\t")
        print(f"Loaded top LVs from {TOP_LV_PATH}")
    else:
        top_lv_rows = []
        for tissue, df_s in shap_store.items():
            top = df_s[df_s["cumulative_pct"] <= CUMULATIVE_PCT]
            if top.empty:
                top = df_s.iloc[[0]]
            top_lv_rows.append(top)

        df_top = pd.concat(top_lv_rows, ignore_index=True)
        df_top.to_csv(TOP_LV_PATH, sep="\t", index=False)
        print(f"Saved top LVs to {TOP_LV_PATH}")

    df_top_unique = (
        df_top.sort_values(["mean_shap", "cumulative_pct"], ascending=[False, True])
        .drop_duplicates("LV", keep="first")
        .sort_values(["tissue", "mean_shap"], ascending=[True, False])
        .reset_index(drop=True)
    )
    df_top_unique.to_csv(UNIQUE_TOP_LV_PATH, sep="\t", index=False)

    top_lvs_by_tissue = {
        tissue: group.sort_values("mean_shap", ascending=False)["LV"].tolist()
        for tissue, group in df_top_unique.groupby("tissue", sort=True)
    }

    print(f"Selected LV rows before de-duplication: {len(df_top)}")
    print(f"Unique selected LVs across tissues    : {df_top_unique['LV'].nunique()}")
    for tissue in passing_tissues:
        lvs = top_lvs_by_tissue.get(tissue, [])
        if lvs:
            print(f"  {tissue:<35s}: {lvs}")


## Biology validation - pathway annotations for top LVs

In [ ]:
FDR_THRESH = 0.01
AUC_THRESH = 0.75

sig_ann = lv_ann[(lv_ann["AUC"] > AUC_THRESH) & (lv_ann["FDR"] < FDR_THRESH)].copy()
print(f"Significant LV-pathway pairs (AUC>{AUC_THRESH}, FDR<{FDR_THRESH}): {len(sig_ann)}")

In [ ]:
if passing_tissues:
    bio_rows = []
    for tissue in passing_tissues:
        top_lvs = top_lvs_by_tissue.get(tissue, [])
        for lv in top_lvs:
            hits = sig_ann[sig_ann["LV"] == lv][["pathway", "AUC", "FDR"]]
            if hits.empty:
                bio_rows.append({"tissue": tissue, "LV": lv,
                                  "pathway": "NO_SIGNIFICANT_PATHWAY",
                                  "AUC": None, "FDR": None})
            else:
                for _, row in hits.iterrows():
                    bio_rows.append({"tissue": tissue, "LV": lv,
                                      "pathway": row["pathway"],
                                      "AUC": row["AUC"], "FDR": row["FDR"]})

    df_bio = pd.DataFrame(bio_rows)
    df_bio.to_csv(BIOLOGY_PATH, sep="\t", index=False)
    print(df_bio.to_string(index=False))


In [ ]:
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)
pd.set_option("display.expand_frame_repr", False)

df_lv_order = (
    df_top_unique[["tissue", "LV", "mean_shap"]]
    .loc[lambda x: x["tissue"].isin(passing_tissues)]
    .sort_values(["tissue", "mean_shap"], ascending=[True, False])
    .assign(
        lv_shap_rank=lambda x: x.groupby("tissue").cumcount() + 1
    )
)

df_lv_pathway_bio = (
    df_lv_order
    .merge(
        sig_ann[["LV", "pathway", "AUC", "FDR"]],
        on="LV",
        how="left"
    )
    .assign(
        pathway=lambda x: x["pathway"].fillna("NO_SIGNIFICANT_PATHWAY")
    )
    .sort_values(
        ["tissue", "lv_shap_rank", "FDR", 'AUC'],
        ascending=[True, True, False, True],
        na_position="last"
    )
    .reset_index(drop=True)
)

df_lv_pathway_bio[["tissue", "LV", "pathway"]]

## Dot plots - pathways and traits for unique selected LVs


In [ ]:
GREEN_CMAP = LinearSegmentedColormap.from_list(
    "biotech_green", ["#d9e6e2", "#4bc17c", "#007a33"]
)

FDR_THRESH_TRAITS = 0.01
MAX_DOTPLOT_ITEMS = 25


def clean_label(text, wrap_width=45):
    if isinstance(text, (pd.Series, list, tuple, np.ndarray)):
        values = pd.Series(text).dropna()
        text = values.iloc[0] if not values.empty else None
    if text is None or pd.isna(text):
        return None
    text = str(text).strip()
    blacklist = ["none of the above", "coffee intake", "job code"]
    if any(term in text.lower() for term in blacklist):
        return None

    text = re.sub(r"\s*\([^)]*?\d+[^)]*?\)", "", text)
    text = text.lstrip("#")
    text = re.sub(r"^(BP|Biological Process)[:\s]*", "", text, flags=re.IGNORECASE)
    if ":" in text:
        text = text.split(":", 1)[1].strip()

    for prefix in ["C2CP_", "REACTOME_", "KEGG_", "WP_", "PID_", "GO_", "HALLMARK_"]:
        if text.upper().startswith(prefix):
            text = text[len(prefix):].strip(" :;,")
            break

    text = re.sub(r"^[A-Z]\d+[\.\d]*\s*", "", text)
    text = text.replace("_", " ").strip()
    if not text or any(term in text.lower() for term in blacklist):
        return None
    text = text[0].upper() + text[1:]
    return "\n".join(textwrap.wrap(text, width=wrap_width, break_long_words=False))


def setup_dotplot_ax(ax, x_label, y_labels, x_lims):
    n = len(y_labels)
    ax.set_facecolor("white")
    ax.set_ylim(n - 0.4, -0.6)
    ax.set_xlabel(x_label, fontsize=14, labelpad=10)
    ax.grid(True, axis="both", linestyle=":", linewidth=0.8, color="#ececec", zorder=0)
    for spine in ["top", "right", "left"]:
        ax.spines[spine].set_visible(False)
    ax.spines["bottom"].set_linewidth(1.0)
    ax.spines["bottom"].set_color("#333333")
    ax.set_yticks(range(n))
    ax.set_yticklabels(y_labels, fontsize=10)
    ax.set_xlim(x_lims)
    ax.tick_params(left=False, labelsize=10)


def unique_label_records(matches, label_col, sort_col, ascending=False, max_items=MAX_DOTPLOT_ITEMS):
    matches = matches.sort_values(sort_col, ascending=ascending)
    records = []
    seen = set()
    for _, row in matches.iterrows():
        label = clean_label(row[label_col])
        if label is None or label in seen:
            continue
        seen.add(label)
        records.append(row.copy())
        if len(records) >= max_items:
            break
    return records


def save_or_show(fig, path=None):
    if path is not None:
        fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()


def plot_pathway_dotplots(tissues, lv_lookup, summary_df, fdr_thresh=0.05, auc_thresh=0.7, top_n=3):
    pathway_dir = DOTPLOT_DIR / f"pathways_top{top_n}_auc"
    pathway_dir.mkdir(exist_ok=True)
    rows = []

    sig = summary_df[(summary_df["AUC"] > auc_thresh) & (summary_df["FDR"] < fdr_thresh)].copy()
    for tissue in tissues:
        lvs = lv_lookup.get(tissue, [])
        if not lvs:
            continue
        matches = sig[sig["LV"].isin(lvs)].copy()
        records = unique_label_records(matches, "pathway", "AUC", ascending=False, max_items=top_n)
        if not records:
            continue

        rec_df = pd.DataFrame(records)
        rec_df.insert(0, "tissue", tissue)
        rec_df.insert(1, "rank_by", "auc")
        rec_df.insert(2, "rank", np.arange(1, len(rec_df) + 1))
        rows.append(rec_df)

        labels = [clean_label(x) for x in rec_df["pathway"]]
        x_vals = rec_df["AUC"].astype(float).to_numpy()
        fdr_logs = np.minimum(3.0, -np.log10(rec_df["FDR"].astype(float).to_numpy()))
        sizes = 100 + ((x_vals - auc_thresh) / max(1e-9, 1 - auc_thresh)) * 500
        sizes = np.clip(sizes, 100, 650)
        x_lims = (max(0.5, auc_thresh - 0.05), 1.0)

        fig_h = max(3.2, 1.7 + 0.55 * len(labels))
        fig = plt.figure(figsize=(11, fig_h), facecolor="white")
        fig.text(0.5, 0.98, tissue.replace(" - ", " - "), ha="center", va="top", fontsize=18)

        cax = fig.add_axes([0.66, 0.84, 0.23, 0.035])
        norm = Normalize(vmin=0, vmax=3)
        sm = plt.cm.ScalarMappable(cmap=GREEN_CMAP, norm=norm)
        cbar = fig.colorbar(sm, cax=cax, orientation="horizontal")
        cbar.outline.set_visible(False)
        cbar.set_ticks([0, 1, 2, 3])
        fig.text(0.65, 0.858, r"$-Log_{10}$ FDR", fontsize=11, ha="right", va="center")

        ax = fig.add_axes([0.42, 0.13, 0.50, 0.63])
        for i, x in enumerate(x_vals):
            ax.plot([x_lims[0], min(x, x_lims[1])], [i, i], color="#cccccc", lw=1, ls=":", zorder=1)
        ax.scatter(
            x_vals,
            range(len(labels)),
            s=sizes,
            c=fdr_logs,
            cmap=GREEN_CMAP,
            norm=norm,
            edgecolor="black",
            linewidth=0.8,
            zorder=3,
            clip_on=False,
        )
        setup_dotplot_ax(ax, "AUC", labels, x_lims)

        out = pathway_dir / f"{tissue.replace('/', '_').replace(' ', '_')}_pathways_top{top_n}_auc.png"
        save_or_show(fig, out)

    if rows:
        df = pd.concat(rows, ignore_index=True)
        df.to_csv(OUT_DIR / f"dotplot_pathway_top{top_n}_auc_data.tsv", sep="\t", index=False)
        return df
    return pd.DataFrame()

def trait_dot_size(n_eff):
    n_eff = pd.to_numeric(n_eff, errors="coerce")
    if pd.isna(n_eff):
        n_eff = 50000
    return float(np.clip(np.sqrt(n_eff) / 1.8, 80, 520))


def plot_trait_dotplots(tissues, lv_lookup, traits_df, fdr_thresh=0.01):
    trait_dir = DOTPLOT_DIR / "traits"
    trait_dir.mkdir(exist_ok=True)
    rows = []

    sig = traits_df[traits_df["fdr"] < fdr_thresh].copy()
    for tissue in tissues:
        lvs = lv_lookup.get(tissue, [])
        if not lvs:
            continue
        matches = sig[sig["LV"].isin(lvs)].copy()
        records = unique_label_records(matches, "phenotype", "fdr", ascending=True)
        if not records:
            continue

        rec_df = pd.DataFrame(records)
        rec_df.insert(0, "tissue", tissue)
        rows.append(rec_df)

        labels = [clean_label(x) for x in rec_df["phenotype"]]
        x_vals = -np.log10(rec_df["fdr"].astype(float).to_numpy())
        v_max = max(3.0, float(np.ceil(x_vals.max())))
        sizes = [trait_dot_size(v) for v in rec_df.get("n_eff", pd.Series([50000] * len(rec_df)))]
        x_lims = (0, v_max + 0.5)

        header_h, footer_h, item_h = 1.25, 0.70, 0.42
        fig_h = max(4.0, header_h + footer_h + item_h * len(labels))
        title_y = 1.0 - (0.18 / fig_h)
        legend_y = 1.0 - (0.72 / fig_h)
        legend_h = 0.26 / fig_h
        ax_bottom = footer_h / fig_h
        ax_height = (item_h * len(labels)) / fig_h

        fig = plt.figure(figsize=(11, fig_h), facecolor="white")
        fig.text(0.5, title_y, tissue.replace(" - ", " - "), ha="center", va="top", fontsize=18)

        lax = fig.add_axes([0.28, legend_y - legend_h / 2, 0.31, legend_h])
        lax.set_xlim(0, 1)
        lax.set_ylim(0, 1)
        lax.axis("off")
        lax.text(0.58, 0.88, "n", ha="center", va="center", fontsize=9)
        size_vals = [50000, 100000, 500000]
        size_x = [0.14, 0.44, 0.73]
        for x_pos, n_val, label in zip(size_x, size_vals, ["50k", "100k", "500k"]):
            lax.scatter(
                [x_pos],
                [0.42],
                s=trait_dot_size(n_val),
                color="#4bc17c",
                edgecolor="black",
                linewidth=0.9,
                zorder=3,
            )
            lax.text(x_pos + 0.085, 0.42, label, ha="left", va="center", fontsize=10)

        cax = fig.add_axes([0.68, legend_y - legend_h / 2, 0.24, legend_h])
        norm = Normalize(vmin=0, vmax=v_max)
        sm = plt.cm.ScalarMappable(cmap=GREEN_CMAP, norm=norm)
        cbar = fig.colorbar(sm, cax=cax, orientation="horizontal")
        cbar.outline.set_visible(False)
        cbar.set_ticks(np.linspace(0, v_max, 4).round(0))
        cax.set_title(r"$-Log_{10}$ FDR", fontsize=11, pad=2)

        ax = fig.add_axes([0.42, ax_bottom, 0.50, ax_height])
        for i, x in enumerate(x_vals):
            ax.plot([x_lims[0], x], [i, i], color="#cccccc", lw=1, ls=":", zorder=1)
        ax.scatter(x_vals, range(len(labels)), s=sizes, c=x_vals, cmap=GREEN_CMAP,
                   norm=norm, edgecolor="black", linewidth=0.8, zorder=3)
        setup_dotplot_ax(ax, r"$-Log_{10}$ FDR", labels, x_lims)

        out = trait_dir / f"{tissue.replace('/', '_').replace(' ', '_')}_traits.png"
        save_or_show(fig, out)

    if rows:
        df = pd.concat(rows, ignore_index=True)
        df.to_csv(OUT_DIR / "dotplot_trait_data.tsv", sep="\t", index=False)
        return df
    return pd.DataFrame()


In [ ]:
df_pathway_top3_auc = plot_pathway_dotplots(
    passing_tissues,
    top_lvs_by_tissue,
    lv_ann,
    fdr_thresh=FDR_THRESH,
    auc_thresh=AUC_THRESH,
    top_n=3,
)
print("Pathways by AUC:", df_pathway_top3_auc.shape)
df_pathway_top3_auc.head()


## Traits from ARCHS4 PhenoMeXcan


In [ ]:
traits = pd.read_csv(TRAITS_PATH, sep="	", compression="gzip", low_memory=False)
traits = traits.rename(columns={"phenotype": "pheno_id", "phenotype_desc": "phenotype", "lv": "LV"})
traits = traits[["pheno_id", "phenotype", "LV", "pvalue", "fdr"]].dropna(
    subset=["phenotype", "LV", "fdr"]
)

pheno_info = pd.read_csv(PHENO_INFO_PATH, sep="	", compression="gzip")
pheno_info["n_eff"] = pheno_info["n"].combine_first(
    pheno_info["n_cases"] + pheno_info["n_controls"]
)
pheno_n_eff = pheno_info.dropna(subset=["short_code", "n_eff"]).set_index("short_code")["n_eff"]
traits["n_eff"] = traits["pheno_id"].map(pheno_n_eff)

traits_selected = traits[traits["LV"].isin(df_top_unique["LV"])].copy()
traits_selected.to_csv(OUT_DIR / "traits_selected_unique_lvs.tsv", sep="	", index=False)

print("Trait associations for unique selected LVs:", traits_selected.shape)
traits_selected.head()


In [ ]:
df_trait_dotplot = plot_trait_dotplots(
    passing_tissues,
    top_lvs_by_tissue,
    traits_selected,
    fdr_thresh=FDR_THRESH_TRAITS,
)
print(df_trait_dotplot.shape)
df_trait_dotplot.head()


## Export for Figure 4

In [ ]:
import shutil
from pyprojroot.here import here as _here

PANEL_DIR_FIG4 = _here() / 'output' / '99_panels' / 'fig4'
PANEL_DIR_FIG4.mkdir(parents=True, exist_ok=True)

# Clean pathway names for fig4 display
def clean_pathway_name(text):
    import re
    text = str(text)
    for prefix in ['HALLMARK_', 'REACTOME_REACTOME_', 'REACTOME_', 'KEGG_', 'WP_',
                   'PID_', 'GO_', 'C2CP_', 'C8_', 'HE_', 'LIM_', 'SUN_']:
        if text.upper().startswith(prefix):
            text = text[len(prefix):]
    text = text.replace('_', ' ').strip()
    text = re.sub(r'\s+', ' ', text)
    return text.title()

# 1. Export biology top LVs with cleaned pathway names (tissue, LV, pathway, AUC, FDR)
df_bio_export = df_bio.copy()
df_bio_export['pathway_clean'] = df_bio_export['pathway'].apply(clean_pathway_name)
df_bio_export.to_csv(PANEL_DIR_FIG4 / 'gtex_tissue_lv_biology.csv', index=False)

# 2. Export top LVs per tissue (unique, deduplicated)
df_top_unique.to_csv(PANEL_DIR_FIG4 / 'gtex_top_lvs_per_tissue.csv', index=False)

# 3. Export RF accuracy summary
df_acc.to_csv(PANEL_DIR_FIG4 / 'gtex_tissue_rf_accuracy.csv', index=False)

print(f"Exported fig4 data to {PANEL_DIR_FIG4}")
print(f"  gtex_tissue_lv_biology.csv    : {df_bio_export.shape}")
print(f"  gtex_top_lvs_per_tissue.csv   : {df_top_unique.shape}")
print(f"  gtex_tissue_rf_accuracy.csv   : {df_acc.shape}")